# Chatbot Especialista com Gemini — NexFlow Desk

O bot usa a seguinte estrutura:
- **Google Gemini** como provedor;
- Carrega a chave de API por `.env`;
- Usa uma base de conhecimento pré-definida fechada como fonte de verdade;
- Recusa informações que não estejam na base;
- Responde exatamente **3 perguntas**;
- Ao final da terceira resposta, gera um **resumo do atendimento** e encerra.

### Domínio escolhido
**NexFlow Desk** é um serviço fictício de suporte gerenciado para pequenas empresas. O procedimento de abertura, prioridade e acompanhamento de chamados abaixo é considerado **informação interna do serviço** e não deve ser complementado com conhecimento externo.


## 1. Instalação das bibliotecas

Execute esta célula uma vez por sessão do Colab.


In [ ]:
!pip -q install -U google-genai python-dotenv


## 2. Carregamento do `.env`

O arquivo `.env` deve ficar na mesma pasta de execução do notebook. No Google Colab, uma forma simples é fazer upload do arquivo.

**Nunca coloque a chave diretamente no notebook ou no repositório.**


In [ ]:
from google.colab import files
from pathlib import Path
import os

# Remove versões anteriores do .env
for arquivo in Path(".").glob(".env*"):
    if arquivo.name.startswith(".env"):
        arquivo.unlink()

print("Selecione o arquivo .env do seu computador:")
uploaded = files.upload()

# Procura o arquivo enviado
arquivos_env = [
    nome for nome in uploaded.keys()
    if nome == ".env"
]

if not arquivos_env:
    raise FileNotFoundError(
        "Nenhum arquivo .env foi enviado. "
        "Envie o arquivo .env, não o .env.example."
    )

print("Arquivo .env carregado com sucesso.")

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

# Try to load .env first, then common renamed versions if the first one doesn't exist
# This handles the case where Colab renames uploaded files due to existing ones.
loaded = False
if Path('.env').exists():
    load_dotenv('.env', override=True)
    loaded = True

if not loaded:
    # If none of the specific files are found, try default load_dotenv()
    # which searches for '.env' in current and parent directories.
    print("Warning: No specific '.env' file found. Trying default load_dotenv().")
    load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-3.6-flash')

if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY não foi encontrada no arquivo .env carregado. Verifique se a chave está presente e correta.')

print(f'Modelo configurado: {GEMINI_MODEL}')
print('API key carregada: OK')

## 3. Inicialização do cliente Gemini

A integração usa o SDK oficial `google-genai`.


In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GOOGLE_API_KEY)


## 4. Personalidade, objetivo, tarefa e base de conhecimento

Neste ponto estão os três elementos de configuração da Pipeline necessária:

1. **Personalidade do atendente**
2. **Objetivo e tarefa**
3. **Conhecimento necessário**

A instrução também define uma regra importante: o modelo só pode usar as informações abaixo para responder sobre o NexFlow Desk.


In [ ]:
SYSTEM_INSTRUCTION = """
Você é a Lara, atendente especialista do NexFlow Desk, um serviço fictício de suporte gerenciado para pequenas empresas.

PERSONALIDADE:
- Educada, profissional, objetiva e clara.
- Explica procedimentos em passos numerados quando isso facilitar a execução.
- Não inventa regras, telas, prazos, valores, contatos, funcionalidades ou procedimentos.

OBJETIVO E TAREFA:
- Responder dúvidas do usuário exclusivamente com base na BASE DE CONHECIMENTO fornecida abaixo.
- A base de conhecimento representa informações internas/privadas do serviço, portanto ela é a única fonte autorizada para as respostas sobre o NexFlow Desk.
- Você pode reorganizar ou resumir o conteúdo da base para facilitar o entendimento, mas não pode acrescentar fatos que não estejam nela.
- Caso a pergunta não possa ser respondida com segurança usando somente a base, responda exatamente:
  "Essa informação não está disponível na base de conhecimento do NexFlow Desk. Por favor, abra um chamado com o suporte."
- Não use conhecimento geral da internet ou conhecimento prévio do modelo para preencher lacunas.
- Não diga que pesquisou a internet. Você não tem permissão para consultar fontes externas nesta tarefa.

BASE DE CONHECIMENTO:

1. Abertura de chamado
- Acesse o Portal NexFlow em portal.nexflow.local
- Entre com seu e-mail corporativo.
- Clique em "Novo chamado".
- Informe: título, descrição do problema, unidade afetada e horário aproximado da ocorrência.
- É recomendado anexar capturas de tela quando houver erro visual.

2. Classificação de prioridade
- Baixa: dúvida, solicitação simples ou problema sem impacto na operação.
- Média: problema que afeta uma parte da operação, mas possui alternativa temporária.
- Alta: problema que interrompe um processo importante para uma equipe.
- Crítica: indisponibilidade geral do serviço NexFlow para a empresa.

3. Alteração de prioridade após abertura
- O solicitante pode pedir alteração de prioridade pelo próprio chamado.
- A equipe de suporte analisa o pedido antes de aplicar a nova prioridade.

4. Acompanhamento do chamado
- Acesse o Portal NexFlow.
- Abra o menu "Meus chamados".
- Selecione o chamado desejado para consultar status, histórico e mensagens da equipe.

5. Reabertura de chamado
- Um chamado encerrado pode ser reaberto quando o mesmo problema voltar a ocorrer.
- Para isso, abra o chamado encerrado em "Meus chamados" e clique em "Reabrir".

6. Informações que NÃO estão disponíveis nesta base
- SLA em horas.
- Telefone do suporte.
- Preços e planos.
- Procedimentos de cancelamento.
- Integrações com outros softwares.
- Detalhes técnicos de infraestrutura.

REGRA FINAL:
Responda somente o que puder ser sustentado pela BASE DE CONHECIMENTO.
"""

contador = 0
encerrado = False
historico = []


## 5. Função de geração de resposta

O histórico é mantido localmente e enviado ao Gemini junto com a instrução de sistema. A temperatura zero reduz variação na resposta, mas a principal proteção contra alucinação é a regra explícita de **fonte única de verdade**.


In [ ]:
def gerar_resposta(mensagens, max_output_tokens=1000):
    """Envia o histórico ao Gemini com limite adequado de saída."""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=mensagens,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0,
            max_output_tokens=max_output_tokens,
            thinking_config=types.ThinkingConfig(
                thinking_budget=0
            ),
        ),
    )

    return response.text.strip() if response.text else "Não foi possível gerar uma resposta."


def responder(pergunta):
    """Processa uma pergunta e controla o limite de 3 respostas."""
    global contador, encerrado, historico

    if encerrado:
        return "Conversa encerrada."

    if contador >= 3:
        encerrado = True
        return "Conversa encerrada."

    contador += 1
    historico.append({"role": "user", "parts": [{"text": pergunta}]})

    resposta = gerar_resposta(historico)
    historico.append({"role": "model", "parts": [{"text": resposta}]})

    if contador == 3:
        resumo_prompt = {
            "role": "user",
            "parts": [{"text": (
                "Agora gere um breve resumo do atendimento com base SOMENTE nas três respostas dadas nesta conversa. "
                "Liste, de forma objetiva, os três assuntos abordados e as orientações fornecidas. "
                "Não acrescente informações novas."
            )}]
        }
        historico.append(resumo_prompt)
        resumo = gerar_resposta(historico, max_output_tokens=500)

        resposta_final = (
            f"{resposta}\n\n"
            f"Resumo do atendimento:\n{resumo}\n\n"
            "Conversa encerrada após 3 perguntas."
        )
        encerrado = True
        return resposta_final

    return resposta


## 6. Executar o chatbot

Digite até três perguntas. Para sair antes do limite, digite `sair`.


In [ ]:
contador = 0
encerrado = False
historico = []

print('Assistente: Olá! Sou a Lara, especialista do NexFlow Desk. Como posso ajudar?')

while not encerrado:
    pergunta_usuario = input('Você: ').strip()

    if pergunta_usuario.lower() == 'sair':
        print('Assistente: Conversa encerrada.')
        encerrado = True
        break

    if not pergunta_usuario:
        print('Assistente: Digite uma pergunta.')
        continue

    resposta_assistente = responder(pergunta_usuario)
    print(f'Assistente: {resposta_assistente}')

print('Sessão finalizada.')



## 7. Roteiro de testes sugerido

Use exatamente três perguntas para demonstrar o requisito principal. Um bom teste mistura perguntas **dentro da base** e uma pergunta **fora da base**, para evidenciar o comportamento anti-alucinação.

Exemplo de roteiro:

1. "Como abro um chamado?"
2. "Qual a prioridade para uma indisponibilidade geral do NexFlow?"
3. "Qual é o SLA em horas para um chamado crítico?"

A terceira pergunta deve receber a mensagem de informação indisponível e, em seguida, o bot deve apresentar o resumo e encerrar.

Lembrando que o usuario sempre pode consultar a base de conhecimento na pipeline para entender o contexto e testar mais perguntas.


## 8. Observação técnica

O controle de três respostas é feito em Python, não pelo LLM. Isso evita depender apenas de uma instrução textual para encerrar a conversa.

Da mesma forma, o resumo é solicitado somente depois de a terceira resposta ser gerada.
